# Smart metering: DLMS/COSEM and M-Bus

Electricity meters speak DLMS/COSEM (IEC 62056); heat and water meters in buildings hang on a wired M-Bus. Here a simulated household meter with rooftop solar and a simulated M-Bus segment stand in for the hardware.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.14.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.14.0-preview.1"

## OBIS codes and COSEM data
Every value in a meter has a six-group logical name; values travel as A-XDR.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Dlms;

var energyCode = ObisCode.Parse("1-0:1.8.0*255");
Console.WriteLine($"{energyCode} = {energyCode.Description}");
var scalerUnit = CosemData.Structure(CosemData.Int8(-1), CosemData.Enum(CosemUnit.Volt));
Console.WriteLine($"{scalerUnit} → {Convert.ToHexString(scalerUnit.Encode())}");

## Read the meter
The public client (SAP 16) associates without a password: it may read, never write. Registers carry a scaler and a unit; the profile is read with selective access by date.

In [ ]:
using IoTCom.Net.Transports;

var meterLink = new InMemoryTransportListener("meter");
await using var meterServer = DlmsServer.Create(o => o.ListenInMemory(meterLink));
await using var householdMeter = new DlmsMeterSimulator(meterServer);
await meterServer.StartAsync();

await using var reader = DlmsClient.Create(o => o.UseInMemory(meterLink));
await reader.ConnectAsync();
foreach (var code in new[] { "1.0.1.8.0.255", "1.0.2.8.0.255", "1.0.32.7.0.255", "1.0.14.7.0.255" })
    Console.WriteLine($"{ObisCode.Parse(code).Description,-34} {await reader.ReadRegisterAsync(ObisCode.Parse(code))}");
var meterNow = await reader.ReadClockAsync();
var lastHour = await reader.ReadProfileAsync(ObisCode.Parse("1.0.99.1.0.255"), meterNow.AddHours(-1), meterNow);
Console.WriteLine($"{lastHour.Rows.Count} load-profile rows in the last hour");

## Write access is earned
The public client cannot open the relay; the management client with a password can.

In [ ]:
// Even with the client-side read-only guard off, the meter refuses the public client.
await using var curious = DlmsClient.Create(o => { o.UseInMemory(meterLink); o.ReadOnly = false; });
await curious.ConnectAsync();
try { await curious.ActionAsync(CosemClass.DisconnectControl, ObisCode.Parse("0.0.96.3.10.255"), 1); }
catch (DlmsException e) { Console.WriteLine($"public client: {e.Message}"); }
await using var engineer = DlmsClient.Create(o => { o.UseInMemory(meterLink).WithPassword("12345678"); o.ReadOnly = false; });
await engineer.ConnectAsync();
await engineer.ActionAsync(CosemClass.DisconnectControl, ObisCode.Parse("0.0.96.3.10.255"), 1);
Console.WriteLine($"relay connected: {householdMeter.Relay.Connected}");
await engineer.ActionAsync(CosemClass.DisconnectControl, ObisCode.Parse("0.0.96.3.10.255"), 2);

## M-Bus: scan a segment
A master pings the primary addresses, then reads each slave's variable data records.

In [ ]:
using IoTCom.Net.Protocols.MBus;

var busLink = new InMemoryTransportListener("mbus");
await using var segment = MBusSlaveSimulator.Create(o => o.ListenInMemory(busLink)).AddDefaultDevices();
await segment.StartAsync();
await using var master = MBusMaster.Create(o => { o.UseInMemory(busLink); o.ResponseTimeout = TimeSpan.FromMilliseconds(100); });
await master.ConnectAsync();
foreach (var address in await master.ScanAsync(0, 5))
{
    var telegram = await master.ReadAsync(address);
    Console.WriteLine($"[{address}] {telegram.SecondaryAddress} {telegram.MediumName}");
    foreach (var record in telegram.Records.Take(3)) Console.WriteLine($"    {record}");
}

## Going further
The Gallery demo *Smart meter reading* draws the household's day from the load profile and the building's M-Bus meters. `iotcom dlms read --sim` and `iotcom mbus scan --sim` do the same in a terminal. See `docs/en/protocols/dlms.md` and `docs/en/protocols/mbus.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*